# Conexão Python + PostgreSQL com Psycopg2

Neste notebook vamos utilizar **Python para se conectar ao PostgreSQL e executar comandos SQL**.

- Conectar o Python ao PostgreSQL com `psycopg2`
- Criar um cursor para executar comandos SQL
- Executar consultas `SELECT`
- Inserir dados com `INSERT`
- Atualizar dados com `UPDATE`
- Excluir dados com `DELETE`
- Utilizar `commit()` para salvar alterações
- Consultar novamente os dados para conferir o resultado
- Encerrar corretamente cursor e conexão

> **Ideia principal:** o SQL continua sendo SQL. O Python passa a ser uma forma de enviar esses comandos ao PostgreSQL e trabalhar com os resultados.

## 1. Biblioteca utilizada

Para conversar com o PostgreSQL a partir do Python, vamos utilizar a biblioteca **Psycopg2**.

Ela funciona como uma ponte entre o programa Python e o banco PostgreSQL.

Fluxo simplificado:

`Python → Psycopg2 → PostgreSQL`

### Instalação

No ambiente de desenvolvimento, podemos instalar a versão pronta para uso com:

Testes:
```bash
pip install psycopg2-binary
```

Em produção:
```bash
pip install psycopg2
```

Depois, no código Python, importamos como `psycopg2`.

In [ ]:
# Execute esta célula para instalar a biblioteca
# !pip install psycopg2

In [1]:
import psycopg2

## 2. Conectando ao PostgreSQL

A função `psycopg2.connect()` cria a conexão com o banco.

Precisamos informar alguns dados da conexão:

| Informação | Exemplo | O que representa |
|---|---|---|
| `host` | `localhost` | Computador onde está o PostgreSQL |
| `database` | `loja_brasil` | Banco de dados que será utilizado |
| `user` | `postgres` | Usuário do PostgreSQL |
| `password` | `postgres` | Senha do usuário |
| `port` | `5432` | Porta padrão do PostgreSQL |

> ⚠️ Os valores abaixo são apenas exemplos. Substitua usuário, senha e banco pelos dados do seu ambiente.

In [24]:
conn = psycopg2.connect(
    host="localhost",
    database="loja_brasil",
    user="postgres",
    password="postgres",
    port=5432,
)

print("Conexão realizada com sucesso!")

Conexão realizada com sucesso!


## 3. Criando o cursor

O objeto **cursor** será utilizado para enviar os comandos SQL ao PostgreSQL.

Podemos pensar nele como o objeto responsável por executar nossas instruções SQL.

In [25]:
cur = conn.cursor()

print("Cursor criado com sucesso!")

Cursor criado com sucesso!


## 4. Executando um SELECT

Agora vamos revisar um comando SQL que já conhecemos.

No SQL puro, escreveríamos:

```sql
SELECT id_cliente, nome, cidade
FROM clientes;
```

Com Psycopg2, colocamos esse comando dentro de `cur.execute()`.

In [26]:
cur.execute("""
    SELECT id_cliente, nome, cidade
    FROM cadastro.clientes;
""")

registros = cur.fetchall()

for registro in registros:
    print(registro)

(2, 'Bruno Henrique Souza', 'Pomerode')
(3, 'Camila Rodrigues', 'Joinville')
(4, 'Daniel Oliveira', 'Itajaí')
(5, 'Eduarda Fernandes', 'Florianópolis')
(6, 'Felipe Almeida', 'Brusque')
(7, 'Gabriela Costa', 'Blumenau')
(8, 'Henrique Martins', 'Jaraguá do Sul')
(9, 'Isabela Santos', 'Rio do Sul')
(10, 'João Pedro Lima', 'Timbó')
(11, 'Karina Souza', 'Gaspar')
(12, 'Lucas Pereira', 'Indaial')
(13, 'Mariana Alves', 'Blumenau')
(14, 'Natália Rocha', 'São José')
(15, 'Otávio Ribeiro', 'Chapecó')
(16, 'Patrícia Mendes', 'Blumenau')
(17, 'Rafael Gomes', 'Pomerode')
(18, 'Sabrina Teixeira', 'Itajaí')
(19, 'Thiago Nunes', 'Balneário Camboriú')
(20, 'Vanessa Cardoso', 'Brusque')
(1, 'Ana Paula Martins', 'Florianópolis')


### O que aconteceu?

- `cur.execute()` enviou o SQL para o PostgreSQL.
- O PostgreSQL executou o `SELECT`.
- `cur.fetchall()` trouxe todas as linhas retornadas.
- O `for` percorreu os registros e exibiu cada linha.

Cada registro retornado pelo Psycopg2 normalmente aparece como uma **tupla Python**.

## 5. SELECT com filtro

Também podemos continuar utilizando os recursos de SQL que já estudamos, como `WHERE`.

Exemplo: consultar apenas clientes de Blumenau.

In [15]:
cur.execute("""
    SELECT id_cliente, nome, cidade
    FROM cadastro.clientes
    WHERE cidade = 'Blumenau';
""")

for registro in cur.fetchall():
    print(registro)

(7, 'Gabriela Costa', 'Blumenau')
(13, 'Mariana Alves', 'Blumenau')
(16, 'Patrícia Mendes', 'Blumenau')


## 6. INSERT — Inserindo dados

Agora vamos utilizar Python para executar um `INSERT`.

O SQL que seria executado no PostgreSQL é:

```sql
INSERT INTO clientes (nome, cidade, email)
VALUES ('João Alves', 'Itajaí', 'joao.alves@email.com');
```

No Python, colocamos esse comando dentro de `cur.execute()`.

In [12]:
cur.execute("""
    INSERT INTO cadastro.clientes (nome, cidade, email, estado, data_cadastro, ativo)
    VALUES ('João Alves', 'Itajaí', 'joao.alves@email.com', 'SC', '2026-09-08', true);
""")

conn.commit()

print("Cliente inserido com sucesso!")

Cliente inserido com sucesso!


### Por que usamos `commit()`?

Comandos que alteram os dados, como `INSERT`, `UPDATE` e `DELETE`, precisam ser **confirmados** para que a alteração seja efetivamente salva no banco.

`conn.commit()` confirma a transação.

Sem o `commit()`, a alteração pode não ser persistida quando a conexão for encerrada.

## 7. Conferindo o INSERT

Depois de inserir, podemos utilizar novamente um `SELECT` para verificar se o registro foi gravado.

In [17]:
cur.execute("""
    SELECT id_cliente, nome, cidade, email
    FROM cadastro.clientes
    WHERE email = 'joao.alves@email.com';
""")

print(cur.fetchone())

None


## 8. UPDATE — Atualizando dados

Podemos utilizar Python para executar um `UPDATE` normalmente.

Neste exemplo, vamos alterar a cidade do cliente com `id_cliente = 1`.

In [14]:
cur.execute("""
    UPDATE cadastro.clientes
    SET cidade = 'Florianópolis'
    WHERE id_cliente = 1;
""")

conn.commit()

print("Cliente atualizado com sucesso!")

Cliente atualizado com sucesso!


## 9. DELETE — Excluindo dados

Da mesma forma, podemos executar um `DELETE` pelo Python.

Para evitar apagar um registro importante da base de aula, vamos excluir o cliente que acabamos de inserir.

In [16]:
cur.execute("""
    DELETE FROM cadastro.clientes
    WHERE email = 'joao.alves@email.com';
""")

conn.commit()

print("Cliente excluído com sucesso!")

Cliente excluído com sucesso!


## 10. Revisando SQL através do Python

O objetivo não é substituir o SQL pelo Python.

O objetivo é perceber que os comandos SQL que aprendemos podem ser **executados programaticamente**.

| SQL | Python + Psycopg2 |
|---|---|
| `SELECT` | `cur.execute()` + `fetchall()` |
| `INSERT` | `cur.execute()` + `commit()` |
| `UPDATE` | `cur.execute()` + `commit()` |
| `DELETE` | `cur.execute()` + `commit()` |

A lógica do SQL continua a mesma. O Python passa a controlar a execução.

## 11. Exemplo: JOIN usando Python

Podemos executar consultas mais completas também.

Por exemplo, revisar um `JOIN` entre clientes e pedidos:

In [22]:
cur.execute("""
    SELECT
        c.nome,
        p.id_pedido,
        p.data_pedido
    FROM cadastro.clientes AS c
    INNER JOIN pedidos AS p
        ON p.id_cliente = c.id_cliente;
""")

for registro in cur.fetchall():
    print(registro)

UndefinedTable: ERRO:  relação "pedidos" não existe
LINE 7:     INNER JOIN pedidos AS p
                       ^


## 12. Exemplo: GROUP BY e agregação

O mesmo vale para funções de agregação.

Podemos revisar `COUNT()` e `GROUP BY` executando a consulta pelo Python.

In [27]:
cur.execute("""
    SELECT
        cidade,
        COUNT(*) AS quantidade_clientes
    FROM cadastro.clientes
    GROUP BY cidade
    ORDER BY quantidade_clientes DESC;
""")

for cidade, quantidade in cur.fetchall():
    print(f"{cidade}: {quantidade} cliente(s)")

Blumenau: 3 cliente(s)
Itajaí: 2 cliente(s)
Brusque: 2 cliente(s)
Pomerode: 2 cliente(s)
Florianópolis: 2 cliente(s)
Balneário Camboriú: 1 cliente(s)
Rio do Sul: 1 cliente(s)
Timbó: 1 cliente(s)
São José: 1 cliente(s)
Jaraguá do Sul: 1 cliente(s)
Joinville: 1 cliente(s)
Chapecó: 1 cliente(s)
Indaial: 1 cliente(s)
Gaspar: 1 cliente(s)


## 13. Tratamento de erro e `rollback()`

Em uma aplicação real, erros podem acontecer durante a execução de um comando.

Nesse caso, podemos utilizar `rollback()` para desfazer alterações ainda não confirmadas.

In [28]:
try:
    cur.execute("""
        INSERT INTO cadastro.clientes (nome, cidade, email)
        VALUES ('Cliente Teste', 'Blumenau', 'teste@email.com');
    """)

    conn.commit()
    print("Operação realizada com sucesso!")

except Exception as erro:
    conn.rollback()
    print("Erro na operação:", erro)

Erro na operação: ERRO:  o valor nulo na coluna "estado" da relação "clientes" viola a restrição de não-nulo
DETAIL:  Registro que falhou contém (23, Cliente Teste, teste@email.com, Blumenau, null, 2026-09-08, t).



## 14. Encerrando a conexão

Ao terminar o programa, devemos fechar o cursor e a conexão.

In [29]:
cur.close()
conn.close()

print("Conexão encerrada com sucesso!")

Conexão encerrada com sucesso!


# Desafio

Agora pratique sozinho:

1. Conecte novamente ao banco.
2. Faça um `SELECT` trazendo clientes de uma cidade específica.
3. Faça um `SELECT` utilizando `ORDER BY`.
4. Insira um novo cliente.
5. Consulte o cliente inserido.
6. Atualize algum dado desse cliente.
7. Consulte novamente para verificar a alteração.
8. Exclua o cliente criado no exercício.
9. Faça um `SELECT` com `COUNT()` e `GROUP BY`.
10. Faça um `JOIN` entre duas tabelas e exiba o resultado.

### Pergunta para reflexão

> O que mudou na consulta SQL quando passamos a executá-la pelo Python?

**Resposta esperada:** a lógica SQL continua praticamente a mesma. O Python passa a ser responsável por enviar o comando ao banco, receber os resultados e utilizar esses dados dentro do programa.

---

## 15. Nova base de teste: tabela de produtos

Nesta seção, vamos criar uma base independente chamada `loja_teste_produtos`. Depois, vamos conectar nela e criar a tabela `public.produtos`.

A tabela possui uma restrição `CHECK` no preço. Ela será usada depois para provocar um erro controlado e observar o efeito do `rollback()`.

In [30]:
import psycopg2
from psycopg2 import sql

nome_banco_teste = "loja_teste_produtos"

conn_admin = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="postgres",
    port=5432,
)
conn_admin.autocommit = True
cur_admin = conn_admin.cursor()

cur_admin.execute(
    "SELECT 1 FROM pg_database WHERE datname = %s",
    (nome_banco_teste,),
)

if cur_admin.fetchone() is None:
    cur_admin.execute(
        sql.SQL("CREATE DATABASE {}")
        .format(sql.Identifier(nome_banco_teste))
    )
    print(f"Banco {nome_banco_teste} criado com sucesso!")
else:
    print(f"Banco {nome_banco_teste} já existe.")

cur_admin.close()
conn_admin.close()

conn_produtos = psycopg2.connect(
    host="localhost",
    database=nome_banco_teste,
    user="postgres",
    password="postgres",
    port=5432,
)

cur_produtos = conn_produtos.cursor()
conn_produtos.autocommit = False

print(f"Conectado ao banco {nome_banco_teste}!")

Banco loja_teste_produtos criado com sucesso!
Conectado ao banco loja_teste_produtos!


In [31]:
cur_produtos.execute("""
    CREATE TABLE IF NOT EXISTS public.produtos (
        id_produto SERIAL PRIMARY KEY,
        codigo VARCHAR(20) UNIQUE NOT NULL,
        nome VARCHAR(100) NOT NULL,
        preco NUMERIC(10, 2) NOT NULL CHECK (preco >= 0),
        estoque INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0)
    );
""")

conn_produtos.commit()
print("Tabela public.produtos criada ou já existente. Commit realizado!")

Tabela public.produtos criada ou já existente. Commit realizado!


## 16. Testando `commit()` com produtos

Neste exemplo, as duas inserções são confirmadas com `commit()`. Depois do commit, uma nova consulta consegue localizar os registros salvos no banco.

ON CONFLIT (codigo) DO UPDATE significa que se a chave já existir, irá atualizar os valores. Existe também o ON CONFLICT (codigo) DO NOTHING que não realiza nenhuma modificação.

In [32]:
cur_produtos.execute("""
    INSERT INTO public.produtos (codigo, nome, preco, estoque)
    VALUES
        ('P001', 'Teclado USB', 89.90, 10),
        ('P002', 'Mouse óptico', 49.90, 25)
    ON CONFLICT (codigo) DO UPDATE
        SET nome = EXCLUDED.nome,
            preco = EXCLUDED.preco,
            estoque = EXCLUDED.estoque;
""")

conn_produtos.commit()
print("Produtos inseridos/atualizados e confirmados com commit()!")

Produtos inseridos/atualizados e confirmados com commit()!


In [33]:
cur_produtos.execute("""
    SELECT id_produto, codigo, nome, preco, estoque
    FROM public.produtos
    WHERE codigo IN ('P001', 'P002')
    ORDER BY id_produto;
""")

for produto in cur_produtos.fetchall():
    print(produto)

(1, 'P001', 'Teclado USB', Decimal('89.90'), 10)
(2, 'P002', 'Mouse óptico', Decimal('49.90'), 25)


## 17. Testando `rollback()` após um erro

O primeiro `INSERT` abaixo é válido, mas ainda não foi confirmado. O segundo viola a regra `CHECK (preco >= 0)`.

Quando o erro acontece, chamamos `rollback()`. Assim, o primeiro `INSERT`, que estava na mesma transação e ainda não tinha `commit()`, também é desfeito.

In [34]:
try:
    cur_produtos.execute("""
        INSERT INTO public.produtos (codigo, nome, preco, estoque)
        VALUES ('TESTE-ROLLBACK', 'Produto temporário', 15.00, 1);
    """)

    cur_produtos.execute("""
        INSERT INTO public.produtos (codigo, nome, preco, estoque)
        VALUES ('TESTE-ERRO', 'Produto inválido', -10.00, 1);
    """)

    conn_produtos.commit()
    print("A transação foi confirmada.")

except psycopg2.Error as erro:
    print("Erro provocado de propósito:", erro.diag.message_primary)
    conn_produtos.rollback()
    print("rollback() executado: alterações não confirmadas foram desfeitas.")

Erro provocado de propósito: a nova linha da relação "produtos" viola a restrição de verificação "produtos_preco_check"
rollback() executado: alterações não confirmadas foram desfeitas.


In [35]:
cur_produtos.execute("""
    SELECT codigo, nome, preco, estoque
    FROM public.produtos
    WHERE codigo IN ('TESTE-ROLLBACK', 'TESTE-ERRO')
    ORDER BY codigo;
""")

produtos_rollback = cur_produtos.fetchall()

print("Resultado após o rollback():")
for produto in produtos_rollback:
    print(produto)

if not produtos_rollback:
    print("Nenhum dos produtos temporários foi salvo.")

Resultado após o rollback():
Nenhum dos produtos temporários foi salvo.


## 18. Encerrando os recursos do teste

Depois de executar as consultas, feche o cursor e a conexão.

In [36]:
cur_produtos.close()
conn_produtos.close()
print("Cursor e conexão dos testes encerrados com sucesso!")

Cursor e conexão dos testes encerrados com sucesso!
